In [1]:
import glob
import json
import numpy as np
import scipy as sp
import networkx as nx
import minorminer
from pathlib import Path
import dimod
import dwave
import dwave.inspector
import os
import pandas as pd
from enum import IntEnum



from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.system import FixedEmbeddingComposite
from dwave.samplers import SteepestDescentSolver

In [2]:
qpu = DWaveSampler(region="eu-central-1")

active_qubits = qpu.nodelist
active_couplers = qpu.edgelist
properties = qpu.properties
chip_id = properties['chip_id']

sampler_properties = {'nodelist': qpu.nodelist, 'edgelist': qpu.edgelist, 'properties': qpu.properties}

sampler_filename = f'sampler_{chip_id}.json'
#with open(sampler_filename, 'w') as f:
#    json.dump(sampler_properties, f, indent=4)


G_qpu = nx.from_edgelist(active_couplers)


In [3]:
class ProblemSize(IntEnum):
    SIZE40 = 40
    SIZE80 = 80
    SIZE100 = 100
    SIZE120 = 120
    SIZE140 = 140
    SIZE160 = 160

    
solution_file = '../../solutions.html'
original_datafile_mask = '../../k-cluster_{}/{}.dat'


def read_solutions_raw(filename):
    solutions = pd.read_html(filename, header=0)[0]
    
    old_columns = ['Instance', 'optimum', 'Instance.1', 'optimum.1', 'Instance.2',
           'optimum.2', 'Instance.3', 'optimum.3', 'Instance.4', 'optimum.4',
           'Instance.5', 'optimum.5']
    
    new_columns = ['instance_40', 'optimum_40', 'instance_80', 'optimum_80', 'instance_100',
           'optimum_100', 'instance_120', 'optimum_120', 'instance_140', 'optimum_140',
           'instance_160', 'optimum_160']

    return solutions.rename(columns=dict(zip(old_columns, new_columns)))

    

def get_A(filename):
    """Read adjacency matrix of a graph and cluster size

    Args:
        filename (_type_): _description_

    Returns:
        raw_data, adjacency_matrix, k
    """
    A_raw = np.loadtxt(filename, dtype=int, usecols=(0, 1))
    n, k = A_raw[0, :]
    
    A = np.zeros(shape=(n,n))
    A[A_raw[1:,0]-1, A_raw[1:,1]-1] = 1
    A = A+A.transpose()

    return A_raw, A, k


def get_solutions(solution_file):
    solutions_row = read_solutions_raw(solution_file)

    def gen_subdict(problem_size):
        for _, (instance, optimum) in solutions_row[[f'instance_{problem_size}', f'optimum_{problem_size}']].iterrows():
            original_datafile = original_datafile_mask.format(problem_size, instance)
            A_raw, A, k = get_A(original_datafile)
            yield instance, {'optimum': optimum, 'k': k, 'A': A}

    return {problem_size: dict(gen_subdict(problem_size)) for problem_size in ProblemSize}

solutions = get_solutions(solution_file)


In [4]:
G_qpu.number_of_nodes()

4516

In [36]:
emb_filebase = f'embedding/{sampler_properties['properties']['chip_id']}'


for problem_size in ProblemSize:
    for instance, data in solutions[problem_size].items():
        print(instance)
        A = data['A']
        G_complement = nx.from_numpy_array(A)
        G = nx.complement(G_complement)

        emb = minorminer.find_embedding(S=G, T=G_qpu)
        emb = {int(k): v for k, v in emb.items()}

        emb_filebase_size = f'{emb_filebase}/{problem_size}'
        Path(emb_filebase_size).mkdir(parents=True, exist_ok=True)
        print(emb_filebase_size)
        #break
    
        file_data_dir_out = f'{emb_filebase_size}/{instance}.json'
        print(file_data_dir_out)
        
    
        with open(file_data_dir_out, 'w') as f:
            json.dump(emb, f, indent=4)


        #break
    #break
    

kcluster40_025_10_1
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster40_025_10_1.json
kcluster40_025_10_2
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster40_025_10_2.json
kcluster40_025_10_3
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster40_025_10_3.json
kcluster40_025_10_4
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster40_025_10_4.json
kcluster40_025_10_5
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster40_025_10_5.json
kcluster40_025_20_1
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster40_025_20_1.json
kcluster40_025_20_2
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster40_025_20_2.json
kcluster40_025_20_3
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster40_025_20_3.json
kcluster40_025_20_4
embedding/Advantage2_system2.1/40
embedding/Advantage2_system2.1/40/kcluster

In [5]:
emb_filebase = f'embedding/{sampler_properties['properties']['chip_id']}/clique'


for problem_size in ProblemSize:
    print(problem_size)
    G = nx.complete_graph(problem_size)

    emb = minorminer.find_embedding(S=G, T=G_qpu)
    emb = {int(k): v for k, v in emb.items()}

    emb_filebase_size = f'{emb_filebase}/{problem_size}'
    Path(emb_filebase_size).mkdir(parents=True, exist_ok=True)
    print(emb_filebase_size)
    #break

    file_data_dir_out = f'{emb_filebase_size}/clique_embedding.json'
    print(file_data_dir_out)
    

    with open(file_data_dir_out, 'w') as f:
        json.dump(emb, f, indent=4)


        #break
    #break
    

40
embedding/Advantage2_system2.1/clique/40
embedding/Advantage2_system2.1/clique/40/clique_embedding.json
80
embedding/Advantage2_system2.1/clique/80
embedding/Advantage2_system2.1/clique/80/clique_embedding.json
100
embedding/Advantage2_system2.1/clique/100
embedding/Advantage2_system2.1/clique/100/clique_embedding.json
120
embedding/Advantage2_system2.1/clique/120
embedding/Advantage2_system2.1/clique/120/clique_embedding.json
140
embedding/Advantage2_system2.1/clique/140
embedding/Advantage2_system2.1/clique/140/clique_embedding.json
160
embedding/Advantage2_system2.1/clique/160
embedding/Advantage2_system2.1/clique/160/clique_embedding.json


In [23]:
solutions[40]

{'kcluster40_025_10_1': {'optimum': 29,
  'k': np.int64(10),
  'A': array([[0., 0., 0., ..., 1., 1., 1.],
         [0., 0., 0., ..., 0., 0., 1.],
         [0., 0., 0., ..., 1., 1., 0.],
         ...,
         [1., 0., 1., ..., 0., 1., 1.],
         [1., 0., 1., ..., 1., 0., 0.],
         [1., 1., 0., ..., 1., 0., 0.]])},
 'kcluster40_025_10_2': {'optimum': 30,
  'k': np.int64(10),
  'A': array([[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 1.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 1.],
         [0., 1., 0., ..., 0., 1., 0.]])},
 'kcluster40_025_10_3': {'optimum': 27,
  'k': np.int64(10),
  'A': array([[0., 0., 0., ..., 0., 0., 1.],
         [0., 0., 1., ..., 0., 1., 1.],
         [0., 1., 0., ..., 1., 0., 0.],
         ...,
         [0., 0., 1., ..., 0., 0., 0.],
         [0., 1., 0., ..., 0., 0., 0.],
         [1., 1., 0., ..., 0., 0., 0.]])},
 'kcluster40_025_10_4': {'optim